In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

Using device: cuda


In [39]:
connection = sqlite3.connect('sales.db')
query = """
SELECT 
    O.Order_ID as Order_ID,
    O.Order_Date,
    U.User_ID as User_ID,
    U.User_Gender,
    U.User_Location,
    P.Product_ID as Product_ID,
    P.Product_Category,
    P.Product_Price,
    O.Return_Status
FROM 
    OrderTable O
LEFT JOIN 
    User U ON O.User_ID = U.User_ID
LEFT JOIN 
    Product P ON O.Product_ID = P.Product_ID;
"""
df = pd.read_sql_query(query, connection)
connection.close()
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price Return_Status  
0  PROD00000000         Clothing         411.59      Returned  
1  PROD00000001            Books         288.88      Returned  
2  PROD00000002             Toys         390.03  Not Returned  
3  PROD00000003             Toys         401.09  Not Returned  
4  PROD00000004            Books         110.09  Not Returned  


In [40]:
df['Returned'] = (df['Return_Status'] == 'Returned').astype(int)
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price Return_Status  Returned  
0  PROD00000000         Clothing         411.59      Returned         1  
1  PROD00000001            Books         288.88      Returned         1  
2  PROD00000002             Toys         390.03  Not Returned         0  
3  PROD00000003             Toys         401.09  Not Returned         0  
4  PROD00000004            Books         110.09  Not Returned         0  


In [41]:
LEAKAGE_COLS = ['Return_Status']
df.drop(columns=LEAKAGE_COLS, inplace=True)
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price  Returned  
0  PROD00000000         Clothing         411.59         1  
1  PROD00000001            Books         288.88         1  
2  PROD00000002             Toys         390.03         0  
3  PROD00000003             Toys         401.09         0  
4  PROD00000004            Books         110.09         0  


In [62]:
df.drop(columns=['Order_ID', 'Product_ID', 'User_ID'])
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price  Returned  
0  PROD00000000         Clothing         411.59         1  
1  PROD00000001            Books         288.88         1  
2  PROD00000002             Toys         390.03         0  
3  PROD00000003             Toys         401.09         0  
4  PROD00000004            Books         110.09         0  


In [63]:
# df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')
# df['Order_Month'] = df['Order_Date'].dt.month.astype(int)
# df['Order_DayOfWeek'] = df['Order_Date'].dt.dayofweek.astype(int)
# df['Order_Quarter'] = df['Order_Date'].dt.quarter.astype(int)
# df['Order_Year'] = df['Order_Date'].dt.year.astype(int)
# df['Is_Weekend'] = df['Order_DayOfWeek'].isin([5, 6]).astype(int)
# season_map = {
#     12: 4, 1: 4, 2: 4,
#     3: 1, 4: 1, 5: 1,
#     6: 2, 7: 2, 8: 2,
#     9: 3, 10: 3, 11: 3
# }
# df['Season'] = df['Order_Month'].map(season_map).astype(int)
# df.drop(columns=['Order_Date'], inplace=True)
# df.dropna()
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price  Returned  
0  PROD00000000         Clothing         411.59         1  
1  PROD00000001            Books         288.88         1  
2  PROD00000002             Toys         390.03         0  
3  PROD00000003             Toys         401.09         0  
4  PROD00000004            Books         110.09         0  


In [64]:
#df['Total_Order_Value'] = (df['Product_Price'] * df['Order_Quantity']).round(4)
#df['Discounted_Price'] = (df['Product_Price'] * (1 - df['Discount_Applied'] / 100)).round(4)
#df['Discount_Amount'] = (df['Product_Price'] - df['Discounted_Price']).round(4)

#median_val = df['Total_Order_Value'].median()
#df['Is_High_Value'] = (df['Total_Order_Value'] > median_val).astype(int)

# OUTLIER_COLS = [
#     'Product_Price',
#     'Order_Quantity',
#     'User_Age',
#     'Discount_Applied',
#     'Total_Order_Value',
#     'Discounted_Price',
#     'Discount_Amount'
# ]

# for col in OUTLIER_COLS:
#     Q1 = df[col].quantile(0.25)
#     Q3 = df[col].quantile(0.75)
#     IQR = Q3 - Q1
#     lower_fence = Q1 - 1.5 * IQR
#     upper_fence = Q3 + 1.5 * IQR
#     df[col] = df[col].clip(lower=lower_fence, upper=upper_fence)

# df['Age_Group'] = pd.cut(
#     df['User_Age'],
#     bins=[0, 25, 35, 50, 100],
#     labels=[0, 1, 2, 3]
# ).astype(int)

# location_freq_map = df['User_Location'].value_counts(normalize=True)
# df['User_Location_Freq'] = df['User_Location'].map(location_freq_map).round(6)

# df.drop(columns=['User_Location','Order_Date'], inplace=True)

# CAT_COLS = ['Product_Category', 'User_Gender', 'Payment_Method', 'Shipping_Method']
# le = LabelEncoder()
# encoding_reference = {}

# for col in CAT_COLS:
#     df[col + '_enc'] = le.fit_transform(df[col].astype(str))
#     encoding_reference[col] = dict(zip(le.classes_, le.transform(le.classes_)))

# df.drop(columns=CAT_COLS, inplace=True)

# if df.isnull().sum().sum() > 0:
#     df.fillna(df.median(numeric_only=True), inplace=True)

# numeric_df = df.select_dtypes(include=[np.number])
# corr_matrix = numeric_df.corr()

# if 'Price_Per_Unit' in df.columns:
#     df.drop(columns=['Price_Per_Unit'], inplace=True)

#OUTPUT_PATH = 'Retail_Dataset_Cleaned.csv'
#df.to_csv(OUTPUT_PATH, index=False)

print("Preprocessing complete")
print(df.shape)

Preprocessing complete
(10000, 9)


In [65]:
print(df.head())

      Order_ID  Order_Date       User_ID User_Gender User_Location  \
0  ORD00000000  05-08-2023  USER00000000        Male        City54   
1  ORD00000001  09-10-2023  USER00000001      Female        City85   
2  ORD00000002  06-05-2023  USER00000002      Female        City30   
3  ORD00000003  29-08-2024  USER00000003        Male        City95   
4  ORD00000004  16-01-2023  USER00000004      Female        City80   

     Product_ID Product_Category  Product_Price  Returned  
0  PROD00000000         Clothing         411.59         1  
1  PROD00000001            Books         288.88         1  
2  PROD00000002             Toys         390.03         0  
3  PROD00000003             Toys         401.09         0  
4  PROD00000004            Books         110.09         0  


In [70]:
categorical_cols = ['Order_Quantity', 'User_Gender_enc', 'Payment_Method_enc', 'Shipping_Method_enc']
continuous_cols = ['Product_Price','Discount_Applied','Total_Order_Value', 'Discounted_Price', 'Discount_Amount', 'User_Location_Freq']

# 3. One-hot encode categorical features and scale continuous features
#df_processed = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

#scaler = StandardScaler()
#df_processed[continuous_cols] = scaler.fit_transform(df_processed[continuous_cols])



# 4. Split into X and y
target_col = 'Returned'
# Force all data to be float32 before extracting the values
X = df.drop(columns=[target_col,"Order_ID","User_ID","Product_ID","Order_Date"]).astype('float32').values
y = df[target_col].astype('float32').values

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

ValueError: could not convert string to float: 'Male'

In [50]:
class ECommerceDataset(Dataset):
    def __init__(self, X, y):
        # Convert data to PyTorch tensors
        self.X = torch.tensor(X, dtype=torch.float32, device=device)
        self.y = torch.tensor(y, dtype=torch.float32, device=device).unsqueeze(1) # Reshape for BCE loss

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Create DataLoaders
train_dataset = ECommerceDataset(X_train, y_train)
test_dataset = ECommerceDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [58]:
class TabularClassifier(nn.Module):
    def __init__(self, input_dim):
        super(TabularClassifier, self).__init__()
        
        # Define the network architecture
        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.ReLU(),
            
            # Final output layer (1 neuron for binary classification)
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

# Initialize the model
input_features = X_train.shape[1]
model = TabularClassifier(input_dim=input_features)
model = model.to(device)
print(model)

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [59]:
# Setup loss function and optimizer
criterion = nn.BCEWithLogitsLoss() # Combines Sigmoid and Binary Cross Entropy
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
epochs = 100
for epoch in range(epochs):
    model.train() # Set model to training mode
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        predictions = model(batch_X)
        
        # 2. Calculate Loss
        loss = criterion(predictions, batch_y)
        
        # 3. Backward pass and optimization
        optimizer.zero_grad() # Clear old gradients
        loss.backward()       # Calculate new gradients
        optimizer.step()      # Update weights
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')

print("Training Complete!")

RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1.
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
